In [ ]:
# Uninstall the mismatched version first
#!pip uninstall torch torchvision torchaudio -y

# Install the build that matches your driver's CUDA version.
# Pick the ONE line that matches what nvidia-smi showed:

# If CUDA 12.1 or 12.2 (most common on Kaggle T4 x2 in 2025):
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

# If CUDA 12.4:
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 -q

# If CUDA 11.8 (P100 sessions):
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
#exit()

In [ ]:
"""
AgroNet - Maturity-Aware Okra Pod Detection Pipeline
"""

import copy
import json
import math
import os
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from pycocotools.coco import COCO
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models as tv_models, transforms

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm

_RESAMPLE_BILINEAR = getattr(getattr(Image, "Resampling", Image), "BILINEAR", 2)
_FLIP_LR = getattr(getattr(Image, "Transpose", Image), "FLIP_LEFT_RIGHT", 0)
_FLIP_TB = getattr(getattr(Image, "Transpose", Image), "FLIP_TOP_BOTTOM", 1)


# ═══════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    """All hyper-parameters in one place."""

    train_sources: List[Tuple[str, str]] = field(default_factory=lambda: [
        ("/kaggle/input/datasets/perrinlitetli/okra-1/images/default",
         "/kaggle/input/datasets/perrinlitetli/okra-1/annotations/instances_default.json"),
        ("/kaggle/input/datasets/perrinlitetli/okra-2/images/default",
         "/kaggle/input/datasets/perrinlitetli/okra-2/annotations/instances_default.json"),
    ])

    train_ratio: float = 0.80
    val_ratio:   float = 0.10
    test_ratio:  float = 0.10
    split_seed:  int   = 42

    img_size:            int  = 640
    pretrained_backbone: bool = True
    reg_max:             int  = 16

    batch_size:           int   = 16
    num_workers:          int   = 4
    epochs:               int   = 1000
    warmup_epochs:        int   = 5
    lr:                   float = 1e-3
    lr_min:               float = 8e-6
    weight_decay:         float = 3e-4
    use_amp:              bool  = True
    backbone_lr_factor:   float = 0.20
    freeze_backbone_epochs: int = 5

    mosaic_prob:      float = 0.80
    mosaic_off_epoch: int   = 60
    mixup_prob:       float = 0.15
    scale_min:        float = 0.70
    scale_max:        float = 1.50
    copy_paste_prob:  float = 0.30

    box_weight: float = 7.5
    cls_weight: float = 15.0
    dfl_weight: float = 2.5
    ord_weight: float = 0.3
    asp_weight: float = 0.2

    cls_loss_reduction: str   = "mean"
    cls_label_smooth:   float = 0.1

    asp_min_ratio: float = 1.5

    tal_alpha:  float = 0.5
    tal_beta:   float = 6.0
    tal_topk:   int   = 10

    conf_threshold:      float = 0.45
    eval_conf_threshold: float = 0.001
    softnms_sigma:       float = 0.3
    softnms_score_thr:   float = 0.001
    iou_threshold:       float = 0.50
    max_det:             int   = 300
    dedup_iou_thr:       float = 0.60
    dedup_centre_px:     float = 35.0

    map_iou_50:    float       = 0.50
    map_iou_range: List[float] = field(default_factory=lambda:
                                       [round(t, 2) for t in
                                        list(np.arange(0.50, 1.00, 0.05))])

    log_dir:     str = "runs/agronet"
    results_dir: str = "results/agronet"

    device: str = "cuda"

    maturity_colors: Dict[str, Tuple[int, int, int]] = field(default_factory=lambda: {
        "immature":    (0,   210, 90),
        "under-mature":(0,   210, 90),
        "under_mature":(0,   210, 90),
        "mature":      (60,  200, 255),
        "good":        (60,  200, 255),
        "over-mature": (255, 120, 0),
        "over_mature": (255, 120, 0),
        "diseased":    (220, 40,  60),
        "damaged":     (220, 40,  60),
    })
    default_bbox_color: Tuple[int, int, int] = (200, 200, 200)


# ═══════════════════════════════════════════════════════════════════════════
# 2. MODEL EMA
# ═══════════════════════════════════════════════════════════════════════════

class ExponentialMovingAverageWeights:
    """Stabilises training by maintaining an EMA copy of model weights."""
    def __init__(self, model: nn.Module, decay: float = 0.9999, tau: int = 2000,
                 updates: int = 0):
        self.ema     = copy.deepcopy(model).eval()
        self.updates = updates
        self.decay   = lambda x: decay * (1 - math.exp(-x / tau))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model: nn.Module) -> None:
        self.updates += 1
        d   = self.decay(self.updates)
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v *= d
                v += (1 - d) * msd[k].detach()


# ═══════════════════════════════════════════════════════════════════════════
# 3. BACKBONE – ResNet-50 (ImageNet pretrained)
# ═══════════════════════════════════════════════════════════════════════════

class ResNet50FeatureExtractor(nn.Module):
    """
    ResNet-50 backbone producing three multi-scale feature maps:
      C3  stride  8   –  512 channels
      C4  stride 16   – 1024 channels
      C5  stride 32   – 2048 channels
    """
    _CONV1_L2_LO: float = 6.0
    _CONV1_L2_HI: float = 12.0

    def __init__(self, pretrained: bool = True):
        super().__init__()
        if pretrained:
            weights = tv_models.ResNet50_Weights.IMAGENET1K_V1
            tqdm.write(f"[Backbone] Loading ResNet-50 weights: {weights}")
        else:
            weights = None
            tqdm.write("[Backbone] Training from scratch (pretrained=False)")

        resnet       = tv_models.resnet50(weights=weights)
        self.conv1   = resnet.conv1
        self.bn1     = resnet.bn1
        self.relu    = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1  = resnet.layer1
        self.layer2  = resnet.layer2
        self.layer3  = resnet.layer3
        self.layer4  = resnet.layer4

        if pretrained:
            self._verify_pretrained_weights()

    def _verify_pretrained_weights(self) -> None:
        with torch.no_grad():
            l2 = self.conv1.weight.norm().item()
        if not (self._CONV1_L2_LO <= l2 <= self._CONV1_L2_HI):
            raise RuntimeError(
                f"[Backbone] Pretrained check FAILED – conv1 L2={l2:.4f} "
                f"not in [{self._CONV1_L2_LO}, {self._CONV1_L2_HI}]")
        rm = self.bn1.running_mean.norm().item()
        if rm == 0.0:
            raise RuntimeError(
                "[Backbone] Pretrained check FAILED – bn1.running_mean is zero")
        tqdm.write(f"[Backbone] ✓ Pretrained verified  conv1_L2={l2:.3f}  bn1_rm={rm:.4f}")

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x  = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x  = self.layer1(x)
        c3 = self.layer2(x)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return c3, c4, c5


# ═══════════════════════════════════════════════════════════════════════════
# 4. BUILDING BLOCKS
# ═══════════════════════════════════════════════════════════════════════════

class ConvBnSilu(nn.Module):
    """Conv2d → BatchNorm → SiLU"""
    def __init__(self, c_in: int, c_out: int, k: int = 3, s: int = 1,
                 p: int = 1, act: bool = True):
        super().__init__()
        self.conv = nn.Conv2d(c_in, c_out, k, stride=s, padding=p, bias=False)
        self.bn   = nn.BatchNorm2d(c_out)
        self.act  = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


class CbamAttentionGate(nn.Module):
    """CBAM-style channel + spatial attention gate."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden            = max(1, channels // reduction)
        self.avg_pool     = nn.AdaptiveAvgPool2d(1)
        self.channel_fc1  = nn.Conv2d(channels, hidden, 1)
        self.channel_fc2  = nn.Conv2d(hidden, channels, 1)
        self.spatial_conv = nn.Conv2d(2, 1, 7, padding=3)
        self.sigmoid      = nn.Sigmoid()
        self.relu         = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ca = self.sigmoid(self.channel_fc2(
             self.relu(self.channel_fc1(self.avg_pool(x)))))
        x  = x * ca
        sa = self.sigmoid(self.spatial_conv(
             torch.cat([x.max(1, keepdim=True)[0],
                        x.mean(1, keepdim=True)], 1)))
        return x * sa


class QKVCrossScaleAttention(nn.Module):
    """Query-Key-Value cross-scale attention."""
    def __init__(self, dim_high: int, dim_low: int, dim_attn: int = 64):
        super().__init__()
        self.q_proj   = nn.Conv2d(dim_high, dim_attn, 1, bias=False)
        self.k_proj   = nn.Conv2d(dim_low,  dim_attn, 1, bias=False)
        self.v_proj   = nn.Conv2d(dim_low,  dim_attn, 1, bias=False)
        self.out_proj = nn.Conv2d(dim_attn, dim_low,  1, bias=False)
        self.scale    = dim_attn ** -0.5

    def forward(self, feat_high: torch.Tensor,
                feat_low: torch.Tensor) -> torch.Tensor:
        B, _, Hh, Wh = feat_high.shape
        _, _, Hl, Wl = feat_low.shape
        q    = self.q_proj(feat_high).view(B, -1, Hh * Wh).permute(0, 2, 1)
        k    = self.k_proj(feat_low).view(B, -1, Hl * Wl)
        v    = self.v_proj(feat_low).view(B, -1, Hl * Wl).permute(0, 2, 1)
        attn = torch.softmax(torch.bmm(q, k) * self.scale, dim=-1)
        out  = torch.bmm(attn, v).permute(0, 2, 1).contiguous().view(B, -1, Hh, Wh)
        out  = F.interpolate(self.out_proj(out), size=(Hl, Wl), mode="nearest")
        return feat_low + out


class ChannelSelectiveFusionCalib(nn.Module):
    """Channel-Selective Fusion Token calibration."""
    def __init__(self, channels_high: int, channels_low: int, reduction: int = 16):
        super().__init__()
        self.align_high = nn.Conv2d(channels_high, channels_low, 1, bias=False)
        hidden          = max(1, channels_low // reduction)
        self.gating_mlp = nn.Sequential(
            nn.Linear(channels_low, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, 2, bias=False),
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, feat_high: torch.Tensor, feat_low: torch.Tensor) -> torch.Tensor:
        f_high = self.align_high(feat_high)
        g      = (F.adaptive_avg_pool2d(f_high, 1).view(f_high.size(0), -1)
                  + F.adaptive_avg_pool2d(feat_low, 1).view(feat_low.size(0), -1))
        w      = self.softmax(self.gating_mlp(g))
        w_h    = w[:, 0].view(-1, 1, 1, 1)
        w_l    = w[:, 1].view(-1, 1, 1, 1)
        feat_high = F.interpolate(f_high, size=feat_low.shape[-2:], mode="nearest")
        return (w_h * feat_high + w_l * feat_low)


class GatedSkipFusionBlock(nn.Module):
    """Learnable-gated raw-image skip connection."""
    def __init__(self, backbone_ch: int, raw_ch: int = 3):
        super().__init__()
        self.raw_proj   = ConvBnSilu(raw_ch, backbone_ch, k=1, s=1, p=0)
        self.gate_conv  = nn.Conv2d(backbone_ch * 2, backbone_ch, 1, bias=True)
        self.gate_act   = nn.Sigmoid()

    def forward(self, backbone_feat: torch.Tensor,
                raw_patch: torch.Tensor) -> torch.Tensor:
        raw_proj = self.raw_proj(
            F.interpolate(raw_patch, size=backbone_feat.shape[2:],
                          mode="bilinear", align_corners=False))
        gate = self.gate_act(
            self.gate_conv(torch.cat([backbone_feat, raw_proj], dim=1)))
        return gate * backbone_feat + (1 - gate) * raw_proj


class SpatialPyramidPoolingFast(nn.Module):
    """SPPF – Spatial Pyramid Pooling Fast."""
    def __init__(self, c_in: int, c_out: int, k: int = 5):
        super().__init__()
        c_ = c_in // 2
        self.reduce  = ConvBnSilu(c_in,   c_,      k=1, s=1, p=0)
        self.project = ConvBnSilu(c_ * 4, c_out,   k=1, s=1, p=0)
        self.pool    = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x  = self.reduce(x)
        y1 = self.pool(x);  y2 = self.pool(y1);  y3 = self.pool(y2)
        return self.project(torch.cat([x, y1, y2, y3], dim=1))


class CrossScaleFPNFusionBlock(nn.Module):
    """FPN merge block with cross-scale fusion."""
    def __init__(self, c_high: int, c_low: int, c_out: int,
                 mode: str = "qkv_cross", use_post_cbam: bool = True):
        super().__init__()
        assert mode in ("qkv_cross", "csft_calib")
        self.proj_high = ConvBnSilu(c_high, c_out, k=1, s=1, p=0)
        self.cross_fuser = (
            QKVCrossScaleAttention(dim_high=c_out, dim_low=c_low)
            if mode == "qkv_cross"
            else ChannelSelectiveFusionCalib(channels_high=c_out, channels_low=c_low))
        self.merge_conv   = ConvBnSilu(c_out + c_low, c_out, k=3, s=1, p=1)
        self.use_post_cbam = use_post_cbam
        if use_post_cbam:
            self.post_cbam = CbamAttentionGate(c_out)

    def forward(self, feat_high: torch.Tensor,
                feat_low: torch.Tensor) -> torch.Tensor:
        feat_high    = self.proj_high(feat_high)
        feat_low_cal = self.cross_fuser(feat_high, feat_low)
        feat_up      = F.interpolate(feat_high, size=feat_low.shape[-2:], mode="nearest")
        x = self.merge_conv(torch.cat([feat_up, feat_low_cal], dim=1))
        if self.use_post_cbam:
            x = self.post_cbam(x)
        return x


# ═══════════════════════════════════════════════════════════════════════════
# 5. DETECTION HEADS
# ═══════════════════════════════════════════════════════════════════════════

class DFLDistributionDecoder(nn.Module):
    """Decodes DFL output at inference time."""
    def __init__(self, reg_max: int = 16):
        super().__init__()
        self.reg_max = reg_max
        self.register_buffer("proj", torch.arange(reg_max, dtype=torch.float))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.softmax(-1) @ self.proj


class MaturityAwareDecoupledHead(nn.Module):
    """Three branches: box, class, maturity."""
    def __init__(self, c_in: int, num_classes: int, reg_max: int = 16,
                 drop_path: float = 0.1):
        super().__init__()
        self.num_classes = num_classes
        self.reg_max     = reg_max
        self.drop_path_p = drop_path

        c_box = max(16, c_in // 4, reg_max * 4)
        c_cls = max(c_in, min(num_classes, 100))
        c_mat = max(16, c_in // 8)

        self.box_branch = nn.Sequential(
            ConvBnSilu(c_in, c_box, 3, 1, 1),
            ConvBnSilu(c_box, c_box, 3, 1, 1),
            nn.Conv2d(c_box, 4 * reg_max, 1),
        )
        self.cls_branch = nn.Sequential(
            ConvBnSilu(c_in, c_cls, 3, 1, 1),
            ConvBnSilu(c_cls, c_cls, 3, 1, 1),
            nn.Conv2d(c_cls, num_classes, 1),
        )
        self.maturity_branch = nn.Sequential(
            ConvBnSilu(c_in, c_mat, 3, 1, 1),
            nn.Conv2d(c_mat, 1, 1),
        )
        self._init_biases()

    def _init_biases(self) -> None:
        prior  = 0.01
        b_init = -math.log((1 - prior) / prior)
        nn.init.constant_(self.cls_branch[-1].bias, b_init)
        nn.init.constant_(self.box_branch[-1].bias, 1.0)

    def _drop_path(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_path_p == 0.0:
            return x
        keep = 1.0 - self.drop_path_p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask  = torch.rand(shape, device=x.device) < keep
        return x * mask / keep

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x_dp = self._drop_path(x)
        return self.box_branch(x_dp), self.cls_branch(x_dp), self.maturity_branch(x_dp)


# ═══════════════════════════════════════════════════════════════════════════
# 6. FULL DETECTOR – AgroNet
# ═══════════════════════════════════════════════════════════════════════════

class AgroNet(nn.Module):
    """
    AgroNet: ResNet-50 + GatedSkipFusion + SPPF + FPN-PAN + MaturityAwareDecoupledHead.
    """
    def __init__(self, num_classes: int, reg_max: int = 16,
                 pretrained_backbone: bool = True):
        super().__init__()
        self.num_classes = num_classes
        self.reg_max     = reg_max
        self.strides     = [8, 16, 32]

        self.backbone = ResNet50FeatureExtractor(pretrained=pretrained_backbone)

        c3, c4, c5 = 512, 1024, 2048

        self.gated_skip_c3 = GatedSkipFusionBlock(backbone_ch=c3)
        self.gated_skip_c4 = GatedSkipFusionBlock(backbone_ch=c4)
        self.gated_skip_c5 = GatedSkipFusionBlock(backbone_ch=c5)

        self.sppf = SpatialPyramidPoolingFast(c5, c5)

        self.fpn_project_p5 = ConvBnSilu(c5, 512, k=1, s=1, p=0)
        self.fpn_fuse_p4    = CrossScaleFPNFusionBlock(c_high=512, c_low=c4, c_out=256, mode="qkv_cross")
        self.fpn_fuse_p3    = CrossScaleFPNFusionBlock(c_high=256, c_low=c3, c_out=128, mode="csft_calib")

        self.panet_downsample_p3 = ConvBnSilu(128, 128, k=3, s=2, p=1)
        self.panet_fuse_p4       = ConvBnSilu(128 + 256, 256, k=3, s=1, p=1)
        self.panet_downsample_p4 = ConvBnSilu(256, 256, k=3, s=2, p=1)
        self.panet_fuse_p5       = ConvBnSilu(256 + 512, 512, k=3, s=1, p=1)

        self.head_p3 = MaturityAwareDecoupledHead(128, num_classes, reg_max, drop_path=0.1)
        self.head_p4 = MaturityAwareDecoupledHead(256, num_classes, reg_max, drop_path=0.1)
        self.head_p5 = MaturityAwareDecoupledHead(512, num_classes, reg_max, drop_path=0.1)

        self.dfl_decoder = DFLDistributionDecoder(reg_max)

    def forward(self, x: torch.Tensor) -> Tuple[List[torch.Tensor], List[torch.Tensor], List[torch.Tensor]]:
        c3, c4, c5 = self.backbone(x)

        c3 = self.gated_skip_c3(c3, x)
        c4 = self.gated_skip_c4(c4, x)
        c5 = self.gated_skip_c5(c5, x)

        c5 = self.sppf(c5)

        p5 = self.fpn_project_p5(c5)
        p4 = self.fpn_fuse_p4(p5, c4)
        p3 = self.fpn_fuse_p3(p4, c3)

        n3 = p3
        n4 = self.panet_fuse_p4(torch.cat([self.panet_downsample_p3(n3), p4], dim=1))
        n5 = self.panet_fuse_p5(torch.cat([self.panet_downsample_p4(n4), p5], dim=1))

        b3, c3_, m3 = self.head_p3(n3)
        b4, c4_, m4 = self.head_p4(n4)
        b5, c5_, m5 = self.head_p5(n5)

        return [b3, b4, b5], [c3_, c4_, c5_], [m3, m4, m5]


# ═══════════════════════════════════════════════════════════════════════════
# 7. ANCHOR UTILITIES
# ═══════════════════════════════════════════════════════════════════════════

def build_anchor_grid(feat_shapes: List[Tuple[int, int]],
                      strides: List[int],
                      device: torch.device,
                      offset: float = 0.5
                      ) -> Tuple[torch.Tensor, torch.Tensor]:
    pts, strd = [], []
    for (h, w), s in zip(feat_shapes, strides):
        xs = torch.arange(w, device=device, dtype=torch.float32) + offset
        ys = torch.arange(h, device=device, dtype=torch.float32) + offset
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        pts.append(torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1))
        strd.append(torch.full((h * w, 1), s, dtype=torch.float32, device=device))
    return torch.cat(pts), torch.cat(strd)


def ltrb_distances_to_xyxy_boxes(ltrb: torch.Tensor,
                                  anchor_points: torch.Tensor) -> torch.Tensor:
    lt, rb = ltrb.chunk(2, dim=-1)
    return torch.cat([anchor_points - lt, anchor_points + rb], dim=-1)


def xyxy_boxes_to_ltrb_distances(anchor_points: torch.Tensor,
                                  gt_xyxy: torch.Tensor,
                                  reg_max: int) -> torch.Tensor:
    lt = anchor_points - gt_xyxy[:, :2]
    rb = gt_xyxy[:, 2:] - anchor_points
    return torch.cat([lt, rb], dim=-1).clamp_(0, reg_max - 1 - 0.01)


# ═══════════════════════════════════════════════════════════════════════════
# 8. TAL LABEL ASSIGNMENT
# ═══════════════════════════════════════════════════════════════════════════

class TaskAlignedLabelAssigner(nn.Module):
    """YOLOv8 Task-Aligned Learning assignment."""
    def __init__(self, top_k: int = 10, alpha: float = 0.5, beta: float = 6.0,
                 eps: float = 1e-9):
        super().__init__()
        self.top_k = top_k
        self.alpha = alpha
        self.beta  = beta
        self.eps   = eps

    @torch.no_grad()
    def forward(
        self,
        cls_preds:     torch.Tensor,
        box_preds:     torch.Tensor,
        anchor_points: torch.Tensor,
        stride_tensor: torch.Tensor,
        gt_boxes:      torch.Tensor,
        gt_labels:     torch.Tensor,
        img_size:      int,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        B, max_gt, _ = gt_boxes.shape
        A            = anchor_points.shape[0]
        nc           = cls_preds.shape[-1]
        device       = cls_preds.device

        anchor_xy     = anchor_points * stride_tensor
        target_boxes  = torch.zeros(B, A, 4,  device=device)
        target_scores = torch.zeros(B, A, nc, device=device)
        fg_mask       = torch.zeros(B, A,     device=device, dtype=torch.bool)
        target_gt_idx = torch.zeros(B, A,     device=device, dtype=torch.long)

        for b in range(B):
            gt_b  = gt_boxes[b];   lbl_b = gt_labels[b]
            valid = lbl_b >= 0;    n_gt  = valid.sum().item()
            if n_gt == 0:
                continue
            gt_b = gt_b[valid];    lbl_b = lbl_b[valid]

            ax  = anchor_xy[:, 0].unsqueeze(0)
            ay  = anchor_xy[:, 1].unsqueeze(0)
            in_box = ((ax > gt_b[:, 0:1]) & (ax < gt_b[:, 2:3]) &
                      (ay > gt_b[:, 1:2]) & (ay < gt_b[:, 3:4]))

            iou_mat = _pairwise_iou_xyxy(box_preds[b].unsqueeze(0),
                                          gt_b.unsqueeze(1))

            cls_gt = cls_preds[b, :, lbl_b].permute(1, 0)
            align  = ((cls_gt.clamp(0, 1) ** self.alpha) *
                      (iou_mat.clamp(0, 1) ** self.beta) *
                      in_box.float())

            topk_vals, _ = align.topk(min(self.top_k, A), dim=1)
            mask_topk    = (align >= topk_vals[:, -1:]) & in_box

            if n_gt > 1:
                conflict = mask_topk.sum(0) > 1
                if conflict.any():
                    best_gt  = align[:, conflict].argmax(0)
                    resolved = torch.zeros(n_gt, conflict.sum(), device=device)
                    resolved.scatter_(0, best_gt.unsqueeze(0), 1)
                    mask_topk[:, conflict] = resolved.bool()

            assigned_gt = mask_topk.float().argmax(0)
            is_fg       = mask_topk.any(0)

            if is_fg.any():
                align_fg   = align[:, is_fg]
                gt_iou_fg  = iou_mat[:, is_fg]
                max_align  = align_fg.amax(0, keepdim=True).clamp(min=self.eps)
                max_iou    = (gt_iou_fg * mask_topk[:, is_fg].float()).amax(0, keepdim=True)
                soft_score = (align_fg / max_align * max_iou).amax(0)

                pos_gt_idx = assigned_gt[is_fg]
                pos_labels = lbl_b[pos_gt_idx]
                soft_cls   = torch.zeros(is_fg.sum(), nc, device=device)
                soft_cls.scatter_(1, pos_labels.unsqueeze(1), soft_score.unsqueeze(1))

                fg_mask[b]               = is_fg
                target_gt_idx[b, is_fg]  = pos_gt_idx
                target_boxes[b, is_fg]   = gt_b[pos_gt_idx]
                target_scores[b, is_fg]  = soft_cls

        return target_boxes, target_scores, fg_mask, target_gt_idx


def _pairwise_iou_xyxy(boxes1: torch.Tensor, boxes2: torch.Tensor,
                        eps: float = 1e-7) -> torch.Tensor:
    x1 = torch.max(boxes1[..., 0], boxes2[..., 0])
    y1 = torch.max(boxes1[..., 1], boxes2[..., 1])
    x2 = torch.min(boxes1[..., 2], boxes2[..., 2])
    y2 = torch.min(boxes1[..., 3], boxes2[..., 3])
    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    a1    = ((boxes1[...,2]-boxes1[...,0])*(boxes1[...,3]-boxes1[...,1])).clamp(0)
    a2    = ((boxes2[...,2]-boxes2[...,0])*(boxes2[...,3]-boxes2[...,1])).clamp(0)
    return inter / (a1 + a2 - inter + eps)


# ═══════════════════════════════════════════════════════════════════════════
# 9. LOSS FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def _ciou_loss(pred_xyxy: torch.Tensor, tgt_xyxy: torch.Tensor,
               eps: float = 1e-7) -> torch.Tensor:
    px1,py1,px2,py2 = pred_xyxy[:,0],pred_xyxy[:,1],pred_xyxy[:,2],pred_xyxy[:,3]
    tx1,ty1,tx2,ty2 = tgt_xyxy[:,0], tgt_xyxy[:,1], tgt_xyxy[:,2], tgt_xyxy[:,3]

    inter_w = (torch.min(px2,tx2)-torch.max(px1,tx1)).clamp(0)
    inter_h = (torch.min(py2,ty2)-torch.max(py1,ty1)).clamp(0)
    inter   = inter_w * inter_h
    pw, ph  = (px2-px1).clamp(0), (py2-py1).clamp(0)
    tw, th  = (tx2-tx1).clamp(0), (ty2-ty1).clamp(0)
    union   = pw*ph + tw*th - inter + eps
    iou     = inter / union

    d2 = ((px1+px2)/2-(tx1+tx2)/2)**2 + ((py1+py2)/2-(ty1+ty2)/2)**2
    enc_w  = (torch.max(px2,tx2)-torch.min(px1,tx1)).clamp(0)
    enc_h  = (torch.max(py2,ty2)-torch.min(py1,ty1)).clamp(0)
    c2     = enc_w**2 + enc_h**2 + eps
    v      = (4/math.pi**2)*(torch.atan(tw/(th+eps))-torch.atan(pw/(ph+eps)))**2
    with torch.no_grad():
        alpha_v = v / (1 - iou + v + eps)
    return 1 - (iou - d2/c2 - alpha_v*v)


class DistributionFocalLoss(nn.Module):
    """DFL for one LTRB distance component."""
    def __init__(self, reg_max: int = 16):
        super().__init__()
        self.reg_max = reg_max

    def forward(self, pred_dist: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        target = target.clamp_(0, self.reg_max - 1 - 0.01)
        tl  = target.long()
        tr  = tl + 1
        wl  = tr.float() - target
        return (F.cross_entropy(pred_dist, tl, reduction="none") * wl
              + F.cross_entropy(pred_dist, tr, reduction="none") * (1.0 - wl))


class OrdinalRankingLoss(nn.Module):
    def __init__(self, ripeness_order: Optional[List[str]] = None,
                 margin: float = 0.2):
        super().__init__()
        self.ripeness_order = ripeness_order
        self.margin         = margin

    def _get_rank(self, label: int, class_names: List[str]) -> int:
        if self.ripeness_order is None:
            return label
        if label >= len(class_names):
            return label
        name = class_names[label].lower().strip()
        for rank, order_name in enumerate(self.ripeness_order):
            o = order_name.lower().strip()
            name_clean = name.replace("-", "").replace("_", "")
            o_clean    = o.replace("-", "").replace("_", "")
            if o in name or name in o or o_clean in name_clean or name_clean in o_clean:
                return rank
        return label

    def forward(self, maturity_scores: torch.Tensor,
                labels: torch.Tensor,
                class_names: List[str]) -> torch.Tensor:
        if maturity_scores.numel() < 2:
            return maturity_scores.sum() * 0.0

        ranks  = torch.tensor(
            [self._get_rank(int(l), class_names) for l in labels],
            dtype=torch.float32, device=maturity_scores.device)

        ri = ranks.unsqueeze(1);  rj = ranks.unsqueeze(0)
        si = maturity_scores.unsqueeze(1); sj = maturity_scores.unsqueeze(0)

        pair_mask = ri < rj
        if not pair_mask.any():
            return maturity_scores.sum() * 0.0

        loss = F.relu(self.margin + si - sj)[pair_mask]
        return loss.mean()


class AspectRatioPriorLoss(nn.Module):
    def __init__(self, min_aspect_ratio: float = 1.5,
                 gt_ratio_gate: float = 1.2):
        super().__init__()
        self.min_ratio    = min_aspect_ratio
        self.gt_ratio_gate = gt_ratio_gate

    def forward(self, pred_xyxy: torch.Tensor,
                gt_xyxy: Optional[torch.Tensor] = None,
                iou_weights: Optional[torch.Tensor] = None) -> torch.Tensor:
        if pred_xyxy.numel() == 0:
            return pred_xyxy.sum() * 0.0

        w_pred = (pred_xyxy[:, 2] - pred_xyxy[:, 0]).clamp(min=1e-4)
        h_pred = (pred_xyxy[:, 3] - pred_xyxy[:, 1]).clamp(min=1e-4)
        ratio_pred = h_pred / w_pred

        if gt_xyxy is not None and gt_xyxy.numel() > 0:
            w_gt = (gt_xyxy[:, 2] - gt_xyxy[:, 0]).clamp(min=1e-4)
            h_gt = (gt_xyxy[:, 3] - gt_xyxy[:, 1]).clamp(min=1e-4)
            gt_ratio = h_gt / w_gt
            gate_mask = gt_ratio >= self.gt_ratio_gate
            if not gate_mask.any():
                return pred_xyxy.sum() * 0.0
            ratio_pred   = ratio_pred[gate_mask]
            iou_weights  = iou_weights[gate_mask] if iou_weights is not None else None

        penalty = F.relu(self.min_ratio - ratio_pred)
        if iou_weights is not None:
            penalty = penalty * (1.0 - iou_weights.clamp(0, 1).detach())
        return penalty.mean()


class AgroNetLoss(nn.Module):
    """Combined loss for AgroNet."""
    def __init__(self, cfg: Config, num_classes: int,
                 class_names: Optional[List[str]] = None):
        super().__init__()
        self.num_classes      = num_classes
        self.reg_max          = cfg.reg_max
        self.img_size         = cfg.img_size
        self.box_weight       = cfg.box_weight
        self.cls_weight       = cfg.cls_weight
        self.dfl_weight       = cfg.dfl_weight
        self.ord_weight       = cfg.ord_weight
        self.asp_weight       = cfg.asp_weight
        self.cls_smooth       = cfg.cls_label_smooth
        self.cls_loss_reduction = cfg.cls_loss_reduction
        self.class_names      = class_names or []

        self.tal_assigner  = TaskAlignedLabelAssigner(
            cfg.tal_topk, cfg.tal_alpha, cfg.tal_beta)
        self.dfl_loss      = DistributionFocalLoss(cfg.reg_max)
        self.dfl_decoder   = DFLDistributionDecoder(cfg.reg_max)

        self.ordinal_loss  = OrdinalRankingLoss(
            ripeness_order=[
                "immature", "under-mature", "under_mature", "undermature",
                "mature", "good",
                "over-mature", "over_mature", "overmature",
            ],
            margin=0.2)
        self.aspect_loss   = AspectRatioPriorLoss(
            min_aspect_ratio=cfg.asp_min_ratio,
            gt_ratio_gate=1.2)

    @staticmethod
    def _collate_targets(targets: List[Dict[str, torch.Tensor]],
                          img_size: int, device: torch.device
                          ) -> Tuple[torch.Tensor, torch.Tensor]:
        B      = len(targets)
        max_gt = max(max(t["boxes"].shape[0] for t in targets), 1)
        boxes  = torch.full((B, max_gt, 4), 0.0,  dtype=torch.float32, device=device)
        labels = torch.full((B, max_gt),    -1,   dtype=torch.long,    device=device)
        for i, t in enumerate(targets):
            n = t["boxes"].shape[0]
            if n == 0:
                continue
            cx, cy, w, h = (t["boxes"][:, 0], t["boxes"][:, 1],
                             t["boxes"][:, 2], t["boxes"][:, 3])
            s  = float(img_size)
            boxes[i, :n]  = torch.stack(
                [(cx-w/2)*s, (cy-h/2)*s, (cx+w/2)*s, (cy+h/2)*s], -1).to(device)
            labels[i, :n] = t["labels"].to(device)
        return boxes, labels

    def _smooth_bce_targets(self, tgt_scores: torch.Tensor) -> torch.Tensor:
        eps = self.cls_smooth
        return tgt_scores * (1.0 - eps) + eps / self.num_classes

    def _compute_cls_loss(self, cls_flat: torch.Tensor, tgt_scores_smooth: torch.Tensor, n_pos: torch.Tensor) -> torch.Tensor:
        if self.cls_loss_reduction == "mean":
            return F.binary_cross_entropy_with_logits(
                cls_flat, tgt_scores_smooth, reduction="mean")
        else:
            return F.binary_cross_entropy_with_logits(
                cls_flat, tgt_scores_smooth, reduction="sum") / n_pos

    def forward(
        self,
        box_preds:      List[torch.Tensor],
        cls_preds:      List[torch.Tensor],
        maturity_preds: List[torch.Tensor],
        targets:        List[Dict[str, torch.Tensor]],
    ) -> Dict[str, torch.Tensor]:

        device = box_preds[0].device
        B      = box_preds[0].shape[0]
        strides     = [8, 16, 32]
        feat_shapes = [(p.shape[2], p.shape[3]) for p in box_preds]
        anchor_pts, stride_t = build_anchor_grid(feat_shapes, strides, device)
        A  = anchor_pts.shape[0]
        nc = self.num_classes

        box_flat = torch.cat(
            [p.permute(0,2,3,1).reshape(B,-1,4*self.reg_max) for p in box_preds], 1)
        cls_flat = torch.cat(
            [p.permute(0,2,3,1).reshape(B,-1,nc) for p in cls_preds], 1)
        mat_flat = torch.cat(
            [p.permute(0,2,3,1).reshape(B,-1,1)  for p in maturity_preds], 1)

        with torch.no_grad():
            dfl_flat   = box_flat.view(B, A, 4, self.reg_max)
            ltrb_fm    = (dfl_flat.softmax(-1) *
                          torch.arange(self.reg_max, device=device, dtype=torch.float32)
                         ).sum(-1)
            anchor_img = anchor_pts * stride_t
            pred_xyxy  = ltrb_distances_to_xyxy_boxes(
                ltrb_fm.view(-1,4) * stride_t.repeat(B,1),
                anchor_img.unsqueeze(0).expand(B,-1,-1).reshape(-1,2)
            ).view(B, A, 4)

        gt_boxes, gt_labels = self._collate_targets(targets, self.img_size, device)
        tgt_boxes, tgt_scores, fg_mask, _ = self.tal_assigner(
            cls_flat.detach().sigmoid(), pred_xyxy.detach(),
            anchor_pts, stride_t, gt_boxes, gt_labels, self.img_size)
        n_pos = fg_mask.sum().clamp(min=1).float()

        tgt_scores_smooth = self._smooth_bce_targets(tgt_scores)
        loss_cls = self._compute_cls_loss(cls_flat, tgt_scores_smooth, n_pos)

        loss_box = torch.tensor(0.0, device=device)
        loss_dfl = torch.tensor(0.0, device=device)
        loss_ord = torch.tensor(0.0, device=device)
        loss_asp = torch.tensor(0.0, device=device)

        if fg_mask.any():
            pos_box_pred = box_flat[fg_mask]
            pos_stride   = stride_t.expand(B, -1, 1)[fg_mask]
            pos_anchor   = anchor_pts.unsqueeze(0).expand(B,-1,-1)[fg_mask]
            pos_mat_pred = mat_flat[fg_mask].squeeze(-1)
            pos_gt_xyxy  = tgt_boxes[fg_mask]

            dfl_pos  = pos_box_pred.view(-1, 4, self.reg_max)
            ltrb_pos = (dfl_pos.softmax(-1) *
                        torch.arange(self.reg_max, device=device, dtype=torch.float32)
                       ).sum(-1)
            pred_pos_xyxy = ltrb_distances_to_xyxy_boxes(
                ltrb_pos * pos_stride, pos_anchor * pos_stride)

            ciou     = _ciou_loss(pred_pos_xyxy, pos_gt_xyxy)
            loss_box = ciou.mean()

            with torch.no_grad():
                iou_w = _pairwise_iou_xyxy(
                    pred_pos_xyxy.unsqueeze(1), pos_gt_xyxy.unsqueeze(0)
                ).squeeze(0).clamp(0)
                iou_w = iou_w.diag() if iou_w.dim() == 2 else iou_w

            gt_ltrb = xyxy_boxes_to_ltrb_distances(
                pos_anchor, pos_gt_xyxy / pos_stride, self.reg_max)
            dfl_sum = sum(self.dfl_loss(dfl_pos[:, k, :], gt_ltrb[:, k]).mean()
                          for k in range(4))
            loss_dfl = dfl_sum / 4.0

            if self.class_names:
                pos_labels = tgt_scores[fg_mask].argmax(-1)
                loss_ord = self.ordinal_loss(
                    pos_mat_pred.sigmoid(), pos_labels, self.class_names)

            loss_asp = self.aspect_loss(pred_pos_xyxy,
                                        gt_xyxy=pos_gt_xyxy,
                                        iou_weights=iou_w)

        total = (self.box_weight * loss_box
               + self.cls_weight * loss_cls
               + self.dfl_weight * loss_dfl
               + self.ord_weight * loss_ord
               + self.asp_weight * loss_asp)

        return {"total": total, "box": loss_box, "cls": loss_cls,
                "dfl": loss_dfl, "ord": loss_ord, "asp": loss_asp}


# ═══════════════════════════════════════════════════════════════════════════
# 10. AUGMENTATION (Mosaic, Mixup, Copy-Paste)
# ═══════════════════════════════════════════════════════════════════════════

def _load_sample_raw(dataset: "OkraCocoDataset", idx: int,
                      scale_min: float, scale_max: float
                      ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    info    = dataset.images[idx]
    src_idx = info["src_idx"]
    img     = Image.open(info["file_name"]).convert("RGB")
    w0, h0  = img.size
    boxes, labels = [], []
    for ann in dataset.cocos[src_idx].loadAnns(
            dataset.cocos[src_idx].getAnnIds(imgIds=[info["img_id"]], iscrowd=False)):
        if ann.get("iscrowd", 0) or ann["bbox"][2] <= 1 or ann["bbox"][3] <= 1:
            continue
        x, y, bw, bh = ann["bbox"]
        boxes.append([min(max((x+bw/2)/w0,0.),1.), min(max((y+bh/2)/h0,0.),1.),
                      min(max(bw/w0,1e-6),1.),  min(max(bh/h0,1e-6),1.)])
        labels.append(dataset.local_to_global[src_idx][ann["category_id"]])
    scale = random.uniform(scale_min, scale_max)
    img   = img.resize((max(1,int(w0*scale)), max(1,int(h0*scale))), _RESAMPLE_BILINEAR)
    return (np.array(img, dtype=np.uint8),
            np.array(boxes,  np.float32) if boxes  else np.zeros((0,4), np.float32),
            np.array(labels, np.int64)   if labels else np.zeros((0,),  np.int64))


def build_mosaic4(dataset: "OkraCocoDataset", idx: int, img_size: int,
                  scale_min: float, scale_max: float,
                  color_jitter: transforms.ColorJitter
                  ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    n    = len(dataset.images)
    idxs = [idx] + random.choices(range(n), k=3)
    cx   = random.randint(img_size//4, 3*img_size//4)
    cy   = random.randint(img_size//4, 3*img_size//4)
    placements = [(0,0,cx,cy),(cx,0,img_size,cy),(0,cy,cx,img_size),(cx,cy,img_size,img_size)]
    canvas     = np.full((img_size, img_size, 3), 114, dtype=np.uint8)
    all_boxes, all_labels = [], []

    for i, (x1,y1,x2,y2) in enumerate(placements):
        img_np, boxes, labels = _load_sample_raw(dataset, idxs[i], scale_min, scale_max)
        if random.random() < 0.5:
            img_np = np.array(color_jitter(Image.fromarray(img_np)), dtype=np.uint8)
        h, w = img_np.shape[:2]
        tw, th = x2-x1, y2-y1
        scale  = min(tw/w, th/h)
        rw, rh = max(1,int(w*scale)), max(1,int(h*scale))
        tile   = np.array(Image.fromarray(img_np).resize((rw,rh), _RESAMPLE_BILINEAR))
        ox = [x2-rw, x1, x2-rw, x1][i]
        oy = [y2-rh, y2-rh, y1, y1][i]
        cx1,cy1 = max(ox,0),max(oy,0)
        cx2,cy2 = min(ox+rw,img_size),min(oy+rh,img_size)
        if cx2>cx1 and cy2>cy1:
            canvas[cy1:cy2,cx1:cx2] = tile[cy1-oy:cy2-oy, cx1-ox:cx2-ox]
        if boxes.shape[0] == 0:
            continue
        bx1 = np.clip((boxes[:,0]-boxes[:,2]/2)*rw+ox, 0, img_size)
        by1 = np.clip((boxes[:,1]-boxes[:,3]/2)*rh+oy, 0, img_size)
        bx2 = np.clip((boxes[:,0]+boxes[:,2]/2)*rw+ox, 0, img_size)
        by2 = np.clip((boxes[:,1]+boxes[:,3]/2)*rh+oy, 0, img_size)
        valid = (bx2-bx1>2)&(by2-by1>2)
        if not valid.any(): continue
        bx1,by1,bx2,by2 = bx1[valid],by1[valid],bx2[valid],by2[valid]
        all_boxes.append(np.stack(
            [((bx1+bx2)/2)/img_size, ((by1+by2)/2)/img_size,
             (bx2-bx1)/img_size,     (by2-by1)/img_size], axis=1))
        all_labels.append(labels[valid])

    final_boxes  = np.concatenate(all_boxes,  0) if all_boxes  else np.zeros((0,4),np.float32)
    final_labels = np.concatenate(all_labels, 0) if all_labels else np.zeros((0,), np.int64)
    return canvas, final_boxes, final_labels


def apply_mixup(img1, boxes1, labels1, img2, boxes2, labels2, alpha=0.5):
    r    = random.betavariate(alpha, alpha)
    if img2.shape[:2] != img1.shape[:2]:
        img2 = np.array(Image.fromarray(img2).resize(
            (img1.shape[1], img1.shape[0]), _RESAMPLE_BILINEAR))
    blended = (r*img1.astype(np.float32)+(1-r)*img2.astype(np.float32)
               ).clip(0,255).astype(np.uint8)
    def _cat(a,b): return (np.concatenate([a,b],0) if a.shape[0] and b.shape[0]
                           else (a if a.shape[0] else b))
    return blended, _cat(boxes1, boxes2), _cat(labels1, labels2)


def apply_copy_paste(img, boxes, labels, donor_img, donor_boxes, donor_labels,
                     prob=0.5):
    if donor_boxes.shape[0] == 0:
        return img, boxes, labels
    H, W = img.shape[:2]
    new_boxes, new_labels = list(boxes), list(labels)
    for i in range(donor_boxes.shape[0]):
        if random.random() > prob: continue
        cx,cy,bw,bh = donor_boxes[i]
        dH,dW = donor_img.shape[:2]
        x1=max(0,int((cx-bw/2)*dW)); x2=min(dW,int((cx+bw/2)*dW))
        y1=max(0,int((cy-bh/2)*dH)); y2=min(dH,int((cy+bh/2)*dH))
        if x2-x1<2 or y2-y1<2: continue
        crop = donor_img[y1:y2,x1:x2]
        pw,ph = x2-x1,y2-y1
        if pw>=W or ph>=H: continue
        px=random.randint(0,W-pw); py=random.randint(0,H-ph)
        img[py:py+ph,px:px+pw] = crop
        new_boxes.append([(px+pw/2)/W,(py+ph/2)/H,pw/W,ph/H])
        new_labels.append(int(donor_labels[i]))
    return (img,
            np.array(new_boxes, np.float32) if new_boxes else np.zeros((0,4),np.float32),
            np.array(new_labels,np.int64)   if new_labels else np.zeros((0,),np.int64))


# ═══════════════════════════════════════════════════════════════════════════
# 11. DATASET
# ═══════════════════════════════════════════════════════════════════════════

class OkraCocoDataset(Dataset):
    """Multi-source COCO-style dataset with unified category space."""

    def __init__(self, sources: List[Tuple[str, str]], img_size: int = 640,
                 augment: bool = False, cfg: Optional[Config] = None):
        super().__init__()
        self.img_size      = img_size
        self.augment       = augment
        self.cfg           = cfg
        self.current_epoch = 0
        self.color_jitter  = transforms.ColorJitter(
            brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)
        self.resize        = transforms.Resize((img_size, img_size))
        self.to_tensor     = transforms.ToTensor()
        self.cocos:             List[COCO]             = []
        self.cat_name_to_gid: Dict[str, int]        = {}
        self.gid_to_cat_name: Dict[int, str]        = {}
        self.local_to_global: List[Dict[int, int]] = []
        self.images:          List[Dict]           = []
        self._build_index(sources)

    def _build_index(self, sources):
        for img_root, ann_path in sources:
            coco = COCO(ann_path)
            self.cocos.append(coco)
            l2g: Dict[int, int] = {}
            for cat in coco.loadCats(coco.getCatIds()):
                name = cat["name"]
                if name not in self.cat_name_to_gid:
                    gid = len(self.cat_name_to_gid)
                    self.cat_name_to_gid[name] = gid
                    self.gid_to_cat_name[gid]  = name
                l2g[cat["id"]] = self.cat_name_to_gid[name]
            self.local_to_global.append(l2g)
        for src_idx, (img_root, _) in enumerate(sources):
            for img_id in self.cocos[src_idx].getImgIds():
                info = self.cocos[src_idx].loadImgs(img_id)[0]
                self.images.append({
                    "src_idx":   src_idx, "img_id": img_id,
                    "file_name": os.path.join(img_root, info["file_name"]),
                    "width": info["width"], "height": info["height"],
                })

    @property
    def num_classes(self) -> int:
        return len(self.cat_name_to_gid)

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        cfg = self.cfg
        use_mosaic = (self.augment and cfg is not None
                      and random.random() < cfg.mosaic_prob
                      and self.current_epoch < cfg.epochs - cfg.mosaic_off_epoch)

        if use_mosaic:
            img_np, boxes_np, labels_np = build_mosaic4(
                self, idx, self.img_size,
                cfg.scale_min, cfg.scale_max, self.color_jitter)
            if random.random() < cfg.copy_paste_prob and len(self.images) > 1:
                d_img, d_boxes, d_labels = _load_sample_raw(
                    self, random.randint(0,len(self.images)-1),
                    cfg.scale_min, cfg.scale_max)
                if d_img.shape[:2] != (self.img_size,)*2:
                    d_img = np.array(Image.fromarray(d_img).resize(
                        (self.img_size,)*2, _RESAMPLE_BILINEAR))
                img_np, boxes_np, labels_np = apply_copy_paste(
                    img_np, boxes_np, labels_np, d_img, d_boxes, d_labels, 0.5)
            if random.random() < cfg.mixup_prob:
                m_img, m_boxes, m_labels = build_mosaic4(
                    self, random.randint(0,len(self.images)-1),
                    self.img_size, cfg.scale_min, cfg.scale_max, self.color_jitter)
                img_np, boxes_np, labels_np = apply_mixup(
                    img_np, boxes_np, labels_np, m_img, m_boxes, m_labels)
            img    = Image.fromarray(img_np)
            boxes  = torch.from_numpy(boxes_np)
            labels = torch.from_numpy(labels_np)
            if random.random() < 0.5:
                img = img.transpose(_FLIP_LR)
                if boxes.numel(): boxes[:, 0] = 1.0 - boxes[:, 0]
            if random.random() < 0.3:
                img = img.transpose(_FLIP_TB)
                if boxes.numel(): boxes[:, 1] = 1.0 - boxes[:, 1]
        else:
            info    = self.images[idx]
            src_idx = info["src_idx"]
            img_id  = info["img_id"]
            img     = Image.open(info["file_name"]).convert("RGB")
            w0, h0  = img.size
            box_list, lbl_list = [], []
            for ann in self.cocos[src_idx].loadAnns(
                    self.cocos[src_idx].getAnnIds(imgIds=[img_id], iscrowd=False)):
                if ann.get("iscrowd",0) or ann["bbox"][2]<=1 or ann["bbox"][3]<=1:
                    continue
                x,y,bw,bh = ann["bbox"]
                box_list.append([min(max((x+bw/2)/w0,0.),1.),
                                  min(max((y+bh/2)/h0,0.),1.),
                                  min(max(bw/w0,1e-6),1.),
                                  min(max(bh/h0,1e-6),1.)])
                lbl_list.append(self.local_to_global[src_idx][ann["category_id"]])
            boxes  = (torch.tensor(box_list, dtype=torch.float32)
                      if box_list  else torch.zeros((0,4)))
            labels = (torch.tensor(lbl_list, dtype=torch.long)
                      if lbl_list else torch.zeros((0,), dtype=torch.long))
            if self.augment:
                if random.random() < 0.5: img = self.color_jitter(img)
                if random.random() < 0.5:
                    img = img.transpose(_FLIP_LR)
                    if boxes.numel(): boxes[:,0] = 1.0 - boxes[:,0]
                if random.random() < 0.3:
                    img = img.transpose(_FLIP_TB)
                    if boxes.numel(): boxes[:,1] = 1.0 - boxes[:,1]

        img_id_val = self.images[idx]["img_id"]
        return (self.to_tensor(self.resize(img)),
                {"boxes":    boxes.float(),
                 "labels":   labels.long(),
                 "image_id": torch.tensor([img_id_val], dtype=torch.int64)})


def okra_collate_fn(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]


def make_train_val_test_splits(
    dataset: OkraCocoDataset,
    train_ratio: float, val_ratio: float, seed: int,
) -> Tuple[Dataset, Dataset, Dataset]:
    n   = len(dataset)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_tr = int(n * train_ratio)
    n_vl = int(n * val_ratio)

    def _copy(augment: bool) -> OkraCocoDataset:
        ds               = copy.copy(dataset)
        ds.augment       = augment
        ds.current_epoch = 0
        ds.cfg           = dataset.cfg if augment else None
        return ds

    return (Subset(_copy(True),  idx[:n_tr]),
            Subset(_copy(False), idx[n_tr:n_tr+n_vl]),
            Subset(_copy(False), idx[n_tr+n_vl:]))


# ═══════════════════════════════════════════════════════════════════════════
# 12. INFERENCE – SoftNMS + Post-NMS Deduplication
# ═══════════════════════════════════════════════════════════════════════════

def _softnms_gaussian(boxes: torch.Tensor, scores: torch.Tensor,
                      sigma: float = 0.3,
                      score_thr: float = 0.001
                      ) -> torch.Tensor:
    if boxes.numel() == 0:
        return torch.empty(0, dtype=torch.long, device=boxes.device)

    boxes_np  = boxes.cpu().numpy().astype(np.float32)
    scores_np = scores.cpu().numpy().astype(np.float32)
    n         = len(scores_np)
    order     = np.argsort(-scores_np)
    keep      = []

    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1: break
        rest  = order[1:]
        b1    = boxes_np[i]
        b_rest= boxes_np[rest]

        xx1 = np.maximum(b1[0], b_rest[:,0])
        yy1 = np.maximum(b1[1], b_rest[:,1])
        xx2 = np.minimum(b1[2], b_rest[:,2])
        yy2 = np.minimum(b1[3], b_rest[:,3])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        a1    = (b1[2]-b1[0]) * (b1[3]-b1[1])
        a_r   = ((b_rest[:,2]-b_rest[:,0]) * (b_rest[:,3]-b_rest[:,1]))
        iou   = inter / (a1 + a_r - inter + 1e-8)

        scores_np[rest] *= np.exp(-(iou ** 2) / sigma)
        order = rest[scores_np[rest] > score_thr]
        order = order[np.argsort(-scores_np[order])]

    return torch.tensor(keep, dtype=torch.long, device=boxes.device)


def _post_nms_deduplication(boxes: torch.Tensor, scores: torch.Tensor,
                             labels: torch.Tensor,
                             iou_thr: float = 0.60,
                             centre_px: float = 35.0
                             ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if boxes.shape[0] <= 1:
        return boxes, scores, labels

    boxes_np  = boxes.cpu().numpy()
    scores_np = scores.cpu().numpy()
    labels_np = labels.cpu().numpy()

    cx = (boxes_np[:,0] + boxes_np[:,2]) / 2
    cy = (boxes_np[:,1] + boxes_np[:,3]) / 2

    used = np.zeros(len(boxes_np), dtype=bool)
    keep = []

    order = np.argsort(-scores_np)
    for i in order:
        if used[i]: continue
        keep.append(i)
        dist = np.sqrt((cx - cx[i])**2 + (cy - cy[i])**2)
        for j in order:
            if used[j] or j == i: continue
            if dist[j] > centre_px: continue
            xx1 = max(boxes_np[i,0], boxes_np[j,0])
            yy1 = max(boxes_np[i,1], boxes_np[j,1])
            xx2 = min(boxes_np[i,2], boxes_np[j,2])
            yy2 = min(boxes_np[i,3], boxes_np[j,3])
            inter = max(0,xx2-xx1)*max(0,yy2-yy1)
            a1 = (boxes_np[i,2]-boxes_np[i,0])*(boxes_np[i,3]-boxes_np[i,1])
            a2 = (boxes_np[j,2]-boxes_np[j,0])*(boxes_np[j,3]-boxes_np[j,1])
            iou = inter / (a1 + a2 - inter + 1e-8)
            if iou > iou_thr:
                used[j] = True
        used[i] = True

    keep = np.array(keep, dtype=np.int64)
    idx  = torch.tensor(keep, device=boxes.device)
    return boxes[idx], scores[idx], labels[idx]


def decode_and_postprocess_predictions(
    box_preds:       List[torch.Tensor],
    cls_preds:       List[torch.Tensor],
    maturity_preds:  List[torch.Tensor],
    strides:         List[int],
    reg_max:         int,
    img_size:        int,
    num_classes:     int,
    conf_threshold:  float,
    cfg:             Config,
    device:          torch.device,
) -> List[Dict[str, torch.Tensor]]:
    B = box_preds[0].shape[0]
    feat_shapes  = [(p.shape[2], p.shape[3]) for p in box_preds]
    anchor_pts, stride_t = build_anchor_grid(feat_shapes, strides, device)
    A = anchor_pts.shape[0]

    box_flat = torch.cat(
        [p.permute(0,2,3,1).reshape(B,-1,4*reg_max) for p in box_preds], 1)
    cls_flat = torch.cat(
        [p.permute(0,2,3,1).reshape(B,-1,num_classes) for p in cls_preds], 1)
    mat_flat = torch.cat(
        [p.permute(0,2,3,1).reshape(B,-1,1) for p in maturity_preds], 1)

    dfl_all  = box_flat.view(B, A, 4, reg_max)
    ltrb_fm  = (dfl_all.softmax(-1) *
                torch.arange(reg_max, device=device, dtype=torch.float32)).sum(-1)
    anchor_img = anchor_pts * stride_t
    ltrb_img   = ltrb_fm * stride_t.unsqueeze(0)
    lt, rb     = ltrb_img.chunk(2, dim=-1)
    boxes_all  = torch.cat([anchor_img-lt, anchor_img+rb], -1).clamp(0, img_size)

    conf_all, cls_all = cls_flat.sigmoid().max(dim=-1)
    mat_all = mat_flat.sigmoid().squeeze(-1)

    empty = {"boxes":  torch.zeros((0,4),  device=device),
             "scores": torch.zeros((0,),   device=device),
             "labels": torch.zeros((0,),   dtype=torch.long, device=device),
             "maturity_scores": torch.zeros((0,), device=device)}
    results = []

    for b in range(B):
        mask = conf_all[b] > conf_threshold
        if not mask.any():
            results.append(empty); continue

        bboxes   = boxes_all[b][mask]
        scores   = conf_all[b][mask]
        labels   = cls_all[b][mask]
        mat_sc   = mat_all[b][mask]

        all_keep = []
        for cls_id in labels.unique():
            cls_mask = labels == cls_id
            cls_idx  = cls_mask.nonzero(as_tuple=True)[0]
            keep_cls = _softnms_gaussian(
                bboxes[cls_mask], scores[cls_mask],
                sigma=cfg.softnms_sigma, score_thr=cfg.softnms_score_thr)
            all_keep.append(cls_idx[keep_cls])

        if not all_keep:
            results.append(empty); continue

        keep_idx = torch.cat(all_keep)
        keep_idx = keep_idx[torch.argsort(-scores[keep_idx])][:cfg.max_det]

        bboxes_k = bboxes[keep_idx]
        scores_k = scores[keep_idx]
        labels_k = labels[keep_idx]
        mat_k    = mat_sc[keep_idx]

        bboxes_k, scores_k, labels_k = _post_nms_deduplication(
            bboxes_k, scores_k, labels_k,
            iou_thr=cfg.dedup_iou_thr, centre_px=cfg.dedup_centre_px)
        results.append({"boxes":          bboxes_k,
                         "scores":         scores_k,
                         "labels":         labels_k,
                         "maturity_scores": mat_k[:len(labels_k)]})
    return results


# ═══════════════════════════════════════════════════════════════════════════
# 13. MATURITY-COLOURED BOUNDING BOX DRAWING
# ═══════════════════════════════════════════════════════════════════════════

def _get_maturity_color(class_name: str,
                        cfg: Config) -> Tuple[int, int, int]:
    name_lower = class_name.lower().strip()
    name_clean = name_lower.replace("-", "").replace("_", "")
    if name_lower in cfg.maturity_colors:
        return cfg.maturity_colors[name_lower]
    for key, color in cfg.maturity_colors.items():
        key_clean = key.replace("-", "").replace("_", "")
        if (key in name_lower or name_lower in key or
                key_clean in name_clean or name_clean in key_clean):
            return color
    return cfg.default_bbox_color


def draw_maturity_detections(
    img: Image.Image,
    boxes: torch.Tensor,
    scores: torch.Tensor,
    labels: torch.Tensor,
    maturity_scores: torch.Tensor,
    class_names: List[str],
    cfg: Config,
    save_path: Optional[str] = None,
) -> Image.Image:
    img  = img.convert("RGB")
    draw = ImageDraw.Draw(img)
    img_w, img_h = img.size
    font_size = max(12, min(28, int(min(img_w, img_h) / 38)))
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    boxes_np = boxes.cpu().numpy()
    scores_np = scores.cpu().numpy()
    labels_np = labels.cpu().numpy()
    mat_np    = (maturity_scores.cpu().numpy()
                 if maturity_scores.numel() else np.zeros(len(labels_np)))

    order = np.argsort(-scores_np)

    for rank, pi in enumerate(order):
        box   = boxes_np[pi]
        score = float(scores_np[pi])
        label = int(labels_np[pi])
        mat   = float(mat_np[pi]) if pi < len(mat_np) else 0.0

        x1, y1, x2, y2 = map(int, box.tolist())
        name  = class_names[label] if 0 <= label < len(class_names) else str(label)
        color = _get_maturity_color(name, cfg)

        lw = max(2, int(score * 5))
        draw.rectangle([(x1,y1),(x2,y2)], outline=color, width=lw)

        text = f"{name}: {score:.2f}"
        try:
            bbox_t = draw.textbbox((0,0), text, font=font)
            tw, th = bbox_t[2]-bbox_t[0], bbox_t[3]-bbox_t[1]
        except AttributeError:
            tw, th = draw.textsize(text, font=font)

        pad = 4
        lw_lbl = tw + 2*pad
        lh_lbl = th + 2*pad
        lt = max(0, x1);  rt = min(img_w, x1 + lw_lbl)
        tt = max(0, y1 - lh_lbl - 2);  bt = tt + lh_lbl
        if bt > y1:
            tt = y1 + 1;  bt = tt + lh_lbl
        if rt > img_w: lt = max(0, img_w - lw_lbl); rt = img_w
        if bt > img_h: bt = img_h; tt = max(0, bt - lh_lbl)

        draw.rectangle([(lt,tt),(rt,bt)], fill=color)
        draw.text((lt+pad, tt+pad), text, fill=(255,255,255), font=font)

        bar_w = max(1, rt - lt)
        bar_h = max(3, int(lh_lbl * 0.25))
        bar_t = bt;  bar_b = min(img_h, bt + bar_h)
        filled_w = int(mat * bar_w)
        r_bar = int(255 * min(1.0, 2 * mat))
        g_bar = int(255 * min(1.0, 2 * (1.0 - mat)))
        draw.rectangle([(lt, bar_t),(lt + filled_w, bar_b)],
                        fill=(r_bar, g_bar, 0))

    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        img.save(save_path)
    return img


# ═══════════════════════════════════════════════════════════════════════════
# 14. mAP COMPUTATION
# ═══════════════════════════════════════════════════════════════════════════

def _iou_matrix_np(bp: np.ndarray, bg: np.ndarray) -> np.ndarray:
    if bp.shape[0] == 0 or bg.shape[0] == 0:
        return np.zeros((bp.shape[0], bg.shape[0]), np.float32)
    x1 = np.maximum(bp[:,None,0], bg[None,:,0])
    y1 = np.maximum(bp[:,None,1], bg[None,:,1])
    x2 = np.minimum(bp[:,None,2], bg[None,:,2])
    y2 = np.minimum(bp[:,None,3], bg[None,:,3])
    inter  = np.maximum(0,x2-x1)*np.maximum(0,y2-y1)
    area_p = ((bp[:,2]-bp[:,0])*(bp[:,3]-bp[:,1]))[:,None]
    area_g = ((bg[:,2]-bg[:,0])*(bg[:,3]-bg[:,1]))[None,:]
    return inter / (area_p + area_g - inter + 1e-8)


def _ap_101point(recalls: np.ndarray, precisions: np.ndarray) -> float:
    r = np.concatenate([[0.], recalls, [1.]])
    p = np.concatenate([[0.], precisions, [0.]])
    for i in range(len(p)-2, -1, -1): p[i] = max(p[i], p[i+1])
    return sum(p[r >= rt][0] if (p[r >= rt]).size else 0. for rt in np.linspace(0,1,101)) / 101.


def _compute_ap_at_iou(all_preds, all_gts, num_classes, iou_thresh):
    aps = []
    for cls in range(num_classes):
        sc, tp_list, n_gt = [], [], 0
        for p, g in zip(all_preds, all_gts):
            pm = p["labels"]==cls;  gm = g["labels"]==cls
            pb = p["boxes"][pm];    gb = g["boxes"][gm];  ps = p["scores"][pm]
            n_gt += gb.shape[0]
            if pb.shape[0] == 0: continue
            if gb.shape[0] == 0:
                sc.extend(ps.tolist()); tp_list.extend([0]*len(ps)); continue
            iou_m   = _iou_matrix_np(pb, gb)
            matched = np.zeros(gb.shape[0], bool)
            for pi in np.argsort(-ps):
                best_iou = iou_thresh - 1e-9;  best_gi = -1
                for gi in range(gb.shape[0]):
                    if matched[gi]: continue
                    if iou_m[pi,gi] > best_iou: best_iou = iou_m[pi,gi]; best_gi = gi
                sc.append(float(ps[pi]))
                if best_gi >= 0: matched[best_gi]=True; tp_list.append(1)
                else:            tp_list.append(0)
        if n_gt == 0: continue
        if not sc:    aps.append(0.); continue
        ord_  = np.argsort(-np.array(sc))
        tp_c  = np.cumsum(np.array(tp_list)[ord_])
        fp_c  = np.cumsum(1 - np.array(tp_list)[ord_])
        rec   = tp_c / (n_gt + 1e-8)
        prec  = tp_c / (tp_c + fp_c + 1e-8)
        aps.append(_ap_101point(rec, prec))
    return float(np.mean(aps)) if aps else 0.


def compute_map_metrics(model: nn.Module, loader: DataLoader,
                        num_classes: int, cfg: Config, device: torch.device,
                        max_batches: Optional[int] = None) -> Dict[str, float]:
    model.eval()
    all_preds, all_gts = [], []
    with torch.no_grad():
        for bi, (imgs, tgts) in enumerate(loader):
            if max_batches and bi >= max_batches: break
            imgs = imgs.to(device)
            bp, cp, mp = model(imgs)
            dets = decode_and_postprocess_predictions(
                bp, cp, mp, [8,16,32], cfg.reg_max, cfg.img_size,
                num_classes, cfg.eval_conf_threshold, cfg, device)
            for i, det in enumerate(dets):
                all_preds.append({
                    "boxes":  det["boxes"].cpu().numpy(),
                    "scores": det["scores"].cpu().numpy(),
                    "labels": det["labels"].cpu().numpy().astype(np.int32)})
                bgt = tgts[i]["boxes"].cpu().numpy()
                lgt = tgts[i]["labels"].cpu().numpy().astype(np.int32)
                s   = cfg.img_size
                if bgt.shape[0] > 0:
                    cx,cy,w,h = bgt[:,0],bgt[:,1],bgt[:,2],bgt[:,3]
                    gb = np.stack([(cx-w/2)*s,(cy-h/2)*s,(cx+w/2)*s,(cy+h/2)*s],1)
                else:
                    gb = np.zeros((0,4), np.float32)
                all_gts.append({"boxes": gb.astype(np.float32), "labels": lgt})

    ap_dict = {f"AP@{t:.2f}": _compute_ap_at_iou(all_preds, all_gts, num_classes, t)
               for t in cfg.map_iou_range}
    map50   = _compute_ap_at_iou(all_preds, all_gts, num_classes, cfg.map_iou_50)
    map5095 = float(np.mean(list(ap_dict.values()))) if ap_dict else 0.
    return {"mAP@50": map50, "mAP@50:95": map5095, **ap_dict}


# ═══════════════════════════════════════════════════════════════════════════
# 15. TRAINER
# ═══════════════════════════════════════════════════════════════════════════

class AgroNetTrainer:
    """Training loop with AMP, EMA, differential LR, and backbone freezing."""
    def __init__(self, model: AgroNet, train_ds: Dataset, val_ds: Dataset,
                 cfg: Config, num_classes: int, class_names: List[str]):
        self.cfg         = cfg
        self.device      = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
        self.model       = model.to(self.device)
        self.epochs      = cfg.epochs
        self.log_dir     = cfg.log_dir
        self.class_names = class_names
        os.makedirs(cfg.log_dir, exist_ok=True)

        self.train_loader = DataLoader(
            train_ds, batch_size=cfg.batch_size, shuffle=True,
            num_workers=cfg.num_workers, pin_memory=True, collate_fn=okra_collate_fn)
        self.val_loader   = (
            DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                       num_workers=cfg.num_workers, pin_memory=True,
                       collate_fn=okra_collate_fn)
            if val_ds is not None else None)

        self.criterion = AgroNetLoss(cfg, num_classes, class_names)

        backbone_params = list(self.model.backbone.parameters())
        other_params    = [p for n, p in self.model.named_parameters()
                           if not any(id(p)==id(bp) for bp in backbone_params)]
        self.optimizer = torch.optim.AdamW([
            {"params": backbone_params, "lr": cfg.lr * cfg.backbone_lr_factor},
            {"params": other_params,    "lr": cfg.lr},
        ], weight_decay=cfg.weight_decay)

        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=cfg.epochs, eta_min=cfg.lr_min)
        self.scaler = GradScaler(enabled=cfg.use_amp and torch.cuda.is_available())
        self.ema    = ExponentialMovingAverageWeights(self.model)

        self.history: Dict[str, list] = {k: [] for k in [
            "epoch","lr","train_total","train_box","train_cls","train_dfl",
            "train_ord","train_asp",
            "val_total","val_box","val_cls","val_dfl","val_ord","val_asp",
            "val_map50","val_map5095"]}
        self.start_epoch   = 0
        self.best_val_loss = float("inf")
        self._maybe_resume()

    @property
    def _ckpt_latest(self): return os.path.join(self.log_dir,"checkpoint_latest.pth")
    @property
    def _ckpt_best(self):   return os.path.join(self.log_dir,"checkpoint_best.pth")

    def _save_checkpoint(self, epoch: int, val_loss: Optional[float]) -> None:
        ckpt = {"epoch": epoch, "model_state": self.ema.ema.state_dict(),
                "optimizer_state": self.optimizer.state_dict(),
                "scheduler_state": self.scheduler.state_dict(),
                "scaler_state":    self.scaler.state_dict(),
                "best_val_loss":   self.best_val_loss,
                "history":         self.history}
        torch.save(ckpt, self._ckpt_latest)
        if val_loss is not None and val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            torch.save(ckpt, self._ckpt_best)
            tqdm.write(f"  [Ckpt] ★ New best  val={val_loss:.4f} → {self._ckpt_best}")

    def _maybe_resume(self) -> None:
        if os.path.isfile(self._ckpt_latest):
            ckpt = torch.load(self._ckpt_latest, map_location=self.device)
            self.model.load_state_dict(ckpt["model_state"])
            self.ema.ema.load_state_dict(ckpt["model_state"])
            self.optimizer.load_state_dict(ckpt["optimizer_state"])
            self.scheduler.load_state_dict(ckpt["scheduler_state"])
            if "scaler_state" in ckpt:
                self.scaler.load_state_dict(ckpt["scaler_state"])
            self.start_epoch   = ckpt["epoch"] + 1
            self.best_val_loss = ckpt.get("best_val_loss", float("inf"))
            self.history       = ckpt.get("history", self.history)
            tqdm.write(f"[Resume] epoch {ckpt['epoch']}  best_val={self.best_val_loss:.4f}")
        else:
            tqdm.write("[Resume] starting from scratch")

    def _apply_warmup_lr(self, epoch: int, step: int, n_steps: int) -> None:
        if epoch >= self.cfg.warmup_epochs: return
        frac = (epoch * n_steps + step) / (self.cfg.warmup_epochs * n_steps)
        lr   = self.cfg.lr_min + (self.cfg.lr - self.cfg.lr_min) * frac
        self.optimizer.param_groups[0]["lr"] = lr * self.cfg.backbone_lr_factor
        self.optimizer.param_groups[1]["lr"] = lr

    def _freeze_backbone(self, freeze: bool) -> None:
        for p in self.model.backbone.parameters():
            p.requires_grad_(not freeze)

    def _plot_training_curves(self) -> None:
        if not self.history["epoch"]: return
        epochs = self.history["epoch"]
        for tk, vk, title, fname in [
            ("train_total","val_total","Total Loss",          "loss_total.png"),
            ("train_box",  "val_box",  "Box Loss (CIoU)",     "loss_box.png"),
            ("train_cls",  "val_cls",  "Classification Loss", "loss_cls.png"),
            ("train_dfl",  "val_dfl",  "DFL Loss",            "loss_dfl.png"),
            ("train_ord",  "val_ord",  "Ordinal Ranking Loss","loss_ord.png"),
            ("train_asp",  "val_asp",  "Aspect Ratio Loss",   "loss_asp.png"),
        ]:
            fig, ax = plt.subplots()
            ax.plot(epochs, self.history[tk], label="train", linewidth=1.5)
            vals = [v if v is not None else float("nan") for v in self.history[vk]]
            if any(not math.isnan(v) for v in vals):
                ax.plot(epochs, vals, label="val", linewidth=1.5)
            ax.set(xlabel="epoch", ylabel="loss", title=title)
            ax.legend(); ax.grid(True, alpha=0.4)
            fig.savefig(os.path.join(self.log_dir, fname), dpi=150); plt.close(fig)
        for key, title, fname in [
            ("val_map50",   "Validation mAP@50",    "map50.png"),
            ("val_map5095", "Validation mAP@50:95", "map5095.png"),
        ]:
            vals = [v if v is not None else float("nan") for v in self.history[key]]
            if not any(not math.isnan(v) for v in vals): continue
            fig, ax = plt.subplots()
            ax.plot(epochs, vals, color="darkorange", linewidth=1.5)
            ax.set(xlabel="epoch", ylabel="mAP", title=title, ylim=(0,1))
            ax.grid(True, alpha=0.4)
            fig.savefig(os.path.join(self.log_dir, fname), dpi=150); plt.close(fig)

    def _run_one_epoch(self, loader: DataLoader, is_train: bool, epoch: int,
                       map50=None, map5095=None):
        model = self.model if is_train else self.ema.ema
        model.train() if is_train else model.eval()
        totals = [0.0] * 6
        phase  = "Train" if is_train else "Val  "
        bar    = tqdm(loader, desc=f"Epoch {epoch+1:>3} [{phase}]",
                      unit="batch", dynamic_ncols=True, leave=True)
        ctx = torch.enable_grad() if is_train else torch.no_grad()
        with ctx:
            for step, (imgs, tgts) in enumerate(bar):
                imgs = imgs.to(self.device)
                if is_train:
                    self._apply_warmup_lr(epoch, step, len(loader))
                    self.optimizer.zero_grad()
                with autocast(device_type="cuda",
                              enabled=self.cfg.use_amp and self.device.type=="cuda"):
                    bp, cp, mp = model(imgs)
                    ld = self.criterion(bp, cp, mp, tgts)
                if is_train:
                    self.scaler.scale(ld["total"]).backward()
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), 5.0)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.ema.update(self.model)
                for k_i, k in enumerate(["total","box","cls","dfl","ord","asp"]):
                    totals[k_i] += ld[k].item()
                n = step + 1
                post = {k: f"{totals[i]/n:.4f}"
                        for i,k in enumerate(["loss","box","cls","dfl","ord","asp"])}
                if not is_train and map50 is not None:
                    post["mAP@50"] = f"{map50:.4f}"
                    post["mAP@50:95"] = f"{map5095:.4f}"
                bar.set_postfix(post)
        bar.close()
        n = max(len(loader), 1)
        return tuple(t / n for t in totals)

    def fit(self) -> None:
        epoch_bar = tqdm(range(self.start_epoch, self.epochs),
                         desc="AgroNet Training", unit="epoch",
                         dynamic_ncols=True, leave=True)
        map50 = map5095 = None
        for epoch in epoch_bar:
            self._freeze_backbone(epoch < self.cfg.freeze_backbone_epochs)

            inner = self.train_loader.dataset
            target = inner.dataset if hasattr(inner, "dataset") else inner
            if hasattr(target, "current_epoch"):
                target.current_epoch = epoch

            tr = self._run_one_epoch(self.train_loader, True,  epoch)
            vl = (self._run_one_epoch(self.val_loader,  False, epoch, map50, map5095)
                  if self.val_loader else (None,)*6)

            if self.val_loader:
                tqdm.write(f"\n  [mAP] epoch {epoch+1} …")
                ms    = compute_map_metrics(self.ema.ema, self.val_loader,
                                            self.criterion.num_classes,
                                            self.cfg, self.device)
                map50   = ms["mAP@50"]
                map5095 = ms["mAP@50:95"]
                tqdm.write(f"  [mAP] mAP@50={map50:.4f}  mAP@50:95={map5095:.4f}")

            if epoch >= self.cfg.warmup_epochs:
                self.scheduler.step()
            cur_lr = self.optimizer.param_groups[1]["lr"]

            self.history["epoch"].append(epoch)
            self.history["lr"].append(cur_lr)
            loss_keys = ["train_total","train_box","train_cls","train_dfl","train_ord","train_asp"]
            for k, v in zip(loss_keys, tr): self.history[k].append(v)
            val_keys  = ["val_total","val_box","val_cls","val_dfl","val_ord","val_asp"]
            for k, v in zip(val_keys, vl):  self.history[k].append(v)
            self.history["val_map50"].append(map50)
            self.history["val_map5095"].append(map5095)

            post = {"lr": f"{cur_lr:.1e}", "tr": f"{tr[0]:.4f}"}
            if vl[0]: post["vl"] = f"{vl[0]:.4f}"
            if map50: post["mAP@50"] = f"{map50:.4f}"; post["mAP@50:95"] = f"{map5095:.4f}"
            epoch_bar.set_postfix(post)
            self._save_checkpoint(epoch, val_loss=vl[0])
            self._plot_training_curves()
        epoch_bar.close()


# ═══════════════════════════════════════════════════════════════════════════
# 16. GRAD-CAM
# ═══════════════════════════════════════════════════════════════════════════

class GradCAMVisualiser:
    """Grad-CAM targeting backbone layer4."""
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, "activations", o.detach()))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, "gradients", go[0].detach()))

    def generate_heatmap(self, x: torch.Tensor) -> np.ndarray:
        assert x.shape[0] == 1
        self.model.zero_grad()
        bp, cp, mp = self.model(x)
        target = torch.cat([c.sigmoid().max(-1)[0].reshape(-1) for c in cp]).max()
        target.backward()
        w   = self.gradients.mean((2,3), keepdim=True)
        cam = F.relu((w * self.activations).sum(1, keepdim=True)).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        H, W = x.shape[2], x.shape[3]
        cam  = np.array(Image.fromarray(np.uint8(cam*255)).resize((W,H), _RESAMPLE_BILINEAR))
        return cam.astype(np.float32) / 255.0


def overlay_gradcam_heatmap(img: Image.Image, heatmap: np.ndarray,
                             alpha: float = 0.4) -> Image.Image:
    colored = (plt.get_cmap("jet")(np.clip(heatmap,0,1))[:,:,:3]*255).astype(np.uint8)
    return Image.blend(img.convert("RGB"), Image.fromarray(colored), alpha)


# ═══════════════════════════════════════════════════════════════════════════
# 17. INFERENCER
# ═══════════════════════════════════════════════════════════════════════════

class AgroNetInferencer:
    """Post-training evaluation and visualisation."""
    def __init__(self, model: AgroNet, checkpoint_path: str,
                 class_names: List[str], cfg: Config):
        self.cfg         = cfg
        self.device      = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
        self.model       = model.to(self.device)
        self.class_names = class_names
        self.num_classes = len(class_names)

        assert os.path.isfile(checkpoint_path), f"Not found: {checkpoint_path}"
        ckpt = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt.get("model_state", ckpt))
        self.model.eval()
        tqdm.write(f"[Inferencer] loaded {checkpoint_path}")

        for d in [os.path.join(cfg.results_dir, s)
                  for s in ("images","gradcam","metrics")]:
            os.makedirs(d, exist_ok=True)
        self.vis_dir     = os.path.join(cfg.results_dir, "images")
        self.gradcam_dir = os.path.join(cfg.results_dir, "gradcam")
        self.metrics_dir = os.path.join(cfg.results_dir, "metrics")

        self.preprocess = transforms.Compose([
            transforms.Resize((cfg.img_size,)*2),
            transforms.ToTensor()])
        self.gradcam = GradCAMVisualiser(self.model, self.model.backbone.layer4)

    def _run_decode(self, imgs: torch.Tensor, use_eval_thr: bool = False
                    ) -> List[Dict[str, torch.Tensor]]:
        thr = self.cfg.eval_conf_threshold if use_eval_thr else self.cfg.conf_threshold
        bp, cp, mp = self.model(imgs)
        return decode_and_postprocess_predictions(
            bp, cp, mp, [8,16,32], self.cfg.reg_max, self.cfg.img_size,
            self.num_classes, thr, self.cfg, self.device)

    @staticmethod
    def _iou_np(b1, b2):
        x1=max(b1[0],b2[0]); y1=max(b1[1],b2[1])
        x2=min(b1[2],b2[2]); y2=min(b1[3],b2[3])
        inter=max(0.,x2-x1)*max(0.,y2-y1)
        a1=max(0.,b1[2]-b1[0])*max(0.,b1[3]-b1[1])
        a2=max(0.,b2[2]-b2[0])*max(0.,b2[3]-b2[1])
        return inter/(a1+a2-inter+1e-8)

    def predict_single_image(self, img: Image.Image,
                              save_path: Optional[str] = None
                              ) -> Dict[str, torch.Tensor]:
        t = self.preprocess(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            det = self._run_decode(t)[0]
        if save_path:
            draw_maturity_detections(
                img.resize((self.cfg.img_size,)*2).convert("RGB"),
                det["boxes"], det["scores"], det["labels"],
                det["maturity_scores"], self.class_names, self.cfg, save_path)
        return det

    def evaluate_dataset(self, loader: DataLoader, max_batches=None, prefix="test"):
        self.model.eval()
        tqdm.write("[Eval] Computing mAP …")
        ms      = compute_map_metrics(self.model, loader, self.num_classes,
                                       self.cfg, self.device, max_batches)
        map50   = ms["mAP@50"]; map5095 = ms["mAP@50:95"]
        tqdm.write(f"[Eval] mAP@50={map50:.4f}   mAP@50:95={map5095:.4f}")

        with open(os.path.join(self.metrics_dir, f"{prefix}_map.json"), "w") as f:
            json.dump({k: round(float(v),6) for k,v in ms.items()}, f, indent=2)

        items = [(k,v) for k,v in ms.items() if k.startswith("AP@")]
        if items:
            thrs = [float(k.replace("AP@","")) for k,_ in items]
            aps  = [v for _,v in items]
            fig, ax = plt.subplots(figsize=(8,5))
            ax.plot(thrs, aps, "o-", color="steelblue", linewidth=2)
            ax.axvline(0.5, ls="--", color="red",   alpha=0.7, label=f"mAP@50={map50:.3f}")
            ax.axhline(map5095, ls="--", color="darkorange", alpha=0.7,
                       label=f"mAP@50:95={map5095:.3f}")
            ax.set(xlabel="IoU threshold", ylabel="AP",
                   title=f"AP vs IoU ({prefix})", xlim=(0.45,1.0), ylim=(0,1))
            ax.legend(); ax.grid(True, alpha=0.4)
            fig.tight_layout()
            fig.savefig(os.path.join(self.metrics_dir, f"{prefix}_ap_iou.png"), dpi=150)
            plt.close(fig)

        scores_pc = {c: [] for c in range(self.num_classes)}
        tp_pc     = {c: [] for c in range(self.num_classes)}
        gt_all, pred_all = [], []
        for bi, (imgs, tgts) in enumerate(loader):
            if max_batches and bi >= max_batches: break
            imgs = imgs.to(self.device)
            with torch.no_grad():
                dets = self._run_decode(imgs, use_eval_thr=True)
            for i in range(imgs.shape[0]):
                det   = dets[i]
                bp_np = det["boxes"].cpu().numpy()
                sp_np = det["scores"].cpu().numpy()
                lp_np = det["labels"].cpu().numpy()
                bgt_n = tgts[i]["boxes"].cpu().numpy()
                lgt   = tgts[i]["labels"].cpu().numpy()
                s     = self.cfg.img_size
                bgt   = (np.stack([(bgt_n[:,0]-bgt_n[:,2]/2)*s,
                                    (bgt_n[:,1]-bgt_n[:,3]/2)*s,
                                    (bgt_n[:,0]+bgt_n[:,2]/2)*s,
                                    (bgt_n[:,1]+bgt_n[:,3]/2)*s], 1)
                         if bgt_n.shape[0] else np.zeros((0,4)))
                matched = np.zeros(len(bgt), bool)
                for pi in np.argsort(-sp_np):
                    best_iou = 0.; best_j = -1
                    for j in range(len(bgt)):
                        if matched[j]: continue
                        iou = self._iou_np(bp_np[pi], bgt[j])
                        if iou > best_iou: best_iou=iou; best_j=j
                    lp_i = int(lp_np[pi])
                    if best_j >= 0 and best_iou >= self.cfg.iou_threshold:
                        matched[best_j] = True
                        tp = int(lp_i == int(lgt[best_j]))
                        scores_pc[lp_i].append(float(sp_np[pi]))
                        tp_pc[lp_i].append(tp)
                        gt_all.append(int(lgt[best_j])); pred_all.append(lp_i)
                    else:
                        scores_pc[lp_i].append(float(sp_np[pi])); tp_pc[lp_i].append(0)

        cm  = (confusion_matrix(gt_all, pred_all, labels=list(range(self.num_classes)))
               if gt_all else np.zeros((self.num_classes,)*2, int))
        fig, ax = plt.subplots(figsize=(10,8))
        im = ax.imshow(cm, cmap=plt.cm.Blues); fig.colorbar(im, ax=ax)
        ax.set(xticks=range(self.num_classes), yticks=range(self.num_classes),
               xticklabels=self.class_names, yticklabels=self.class_names,
               ylabel="True", xlabel="Predicted",
               title=f"Confusion Matrix ({prefix})")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        thr = cm.max()/2 if cm.max() else 0.5
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j,i,cm[i,j],ha="center",va="center",fontsize=9,
                        color="white" if cm[i,j]>thr else "black")
        fig.tight_layout()
        fig.savefig(os.path.join(self.metrics_dir,f"{prefix}_confusion_matrix.png"),dpi=150)
        plt.close(fig)

        if gt_all:
            report = classification_report(
                gt_all, pred_all, labels=list(range(self.num_classes)),
                target_names=self.class_names, output_dict=True, zero_division=0)
            with open(os.path.join(self.metrics_dir,f"{prefix}_report.json"),"w") as f:
                json.dump(report, f, indent=2)

        cmap = plt.cm.tab10
        fig_roc, ax_roc = plt.subplots(figsize=(9,7))
        fig_pr,  ax_pr  = plt.subplots(figsize=(9,7))
        ax_roc.plot([0,1],[0,1],"k--", linewidth=0.8, alpha=0.5)
        for c in range(self.num_classes):
            sc = np.array(scores_pc[c], np.float32)
            tp = np.array(tp_pc[c],     np.int32)
            if sc.size == 0: continue
            name  = self.class_names[c]
            color = cmap(c / max(self.num_classes-1, 1))
            try:
                fpr,tpr,_ = roc_curve(tp,sc)
                auc = roc_auc_score(tp,sc) if len(np.unique(tp))>1 else 0.
                ax_roc.plot(fpr,tpr,color=color,linewidth=1.8,
                            label=f"{name} (AUC={auc:.3f})")
            except Exception as e:
                tqdm.write(f"  [ROC] {name}: {e}")
            try:
                prec,rec,_ = precision_recall_curve(tp,sc)
                ap = _ap_101point(rec,prec)
                ax_pr.plot(rec,prec,color=color,linewidth=1.8,
                           label=f"{name} (AP={ap:.3f})")
            except Exception as e:
                tqdm.write(f"  [PR] {name}: {e}")
        ax_roc.set(xlabel="FPR",ylabel="TPR",title=f"ROC – All Classes ({prefix})",
                   xlim=(0,1),ylim=(0,1.02))
        ax_roc.legend(loc="lower right",fontsize=9); ax_roc.grid(True,alpha=0.35)
        fig_roc.tight_layout()
        fig_roc.savefig(os.path.join(self.metrics_dir,f"{prefix}_roc.png"),dpi=150)
        plt.close(fig_roc)
        ax_pr.set(xlabel="Recall",ylabel="Precision",title=f"PR – All Classes ({prefix})",
                  xlim=(0,1),ylim=(0,1.02))
        ax_pr.legend(loc="upper right",fontsize=9); ax_pr.grid(True,alpha=0.35)
        fig_pr.tight_layout()
        fig_pr.savefig(os.path.join(self.metrics_dir,f"{prefix}_pr.png"),dpi=150)
        plt.close(fig_pr)
        tqdm.write(f"[Eval] metrics → {self.metrics_dir}")

    def visualize_dataset(self, loader: DataLoader, max_images: int = 50,
                          prefix: str = "test") -> None:
        out = os.path.join(self.vis_dir, prefix)
        os.makedirs(out, exist_ok=True)
        count = 0
        for imgs, tgts in loader:
            if count >= max_images: break
            imgs = imgs.to(self.device)
            with torch.no_grad():
                dets = self._run_decode(imgs)
            for i in range(imgs.shape[0]):
                if count >= max_images: break
                img_np = (imgs[i].cpu().numpy().transpose(1,2,0)*255
                          ).clip(0,255).astype(np.uint8)
                img_id = int(tgts[i]["image_id"].item()) if "image_id" in tgts[i] else count
                draw_maturity_detections(
                    Image.fromarray(img_np),
                    dets[i]["boxes"], dets[i]["scores"],
                    dets[i]["labels"], dets[i]["maturity_scores"],
                    self.class_names, self.cfg,
                    save_path=os.path.join(out, f"{prefix}_{img_id}.png"))
                count += 1
        tqdm.write(f"[Vis] {count} images → {out}")

    def gradcam_dataset(self, loader: DataLoader, max_images: int = 20,
                        prefix: str = "test") -> None:
        out = os.path.join(self.gradcam_dir, prefix)
        os.makedirs(out, exist_ok=True)
        count = 0
        for imgs, tgts in loader:
            for i in range(imgs.shape[0]):
                if count >= max_images: return
                img_np = (imgs[i].cpu().numpy().transpose(1,2,0)*255
                          ).clip(0,255).astype(np.uint8)
                img_id = int(tgts[i]["image_id"].item()) if "image_id" in tgts[i] else count
                pil    = Image.fromarray(img_np)
                tensor = self.preprocess(pil).unsqueeze(0).to(self.device)
                heatmap = self.gradcam.generate_heatmap(tensor)
                overlay = overlay_gradcam_heatmap(pil, heatmap)
                with torch.no_grad():
                    det = self._run_decode(tensor)[0]
                overlay = draw_maturity_detections(
                    overlay, det["boxes"], det["scores"],
                    det["labels"], det["maturity_scores"],
                    self.class_names, self.cfg)
                overlay.save(os.path.join(out, f"{prefix}_cam_{img_id}.png"))
                count += 1
        tqdm.write(f"[GradCAM] {count} overlays → {out}")


# ═══════════════════════════════════════════════════════════════════════════
# 18. ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    cfg = Config()

    full_ds = OkraCocoDataset(cfg.train_sources, cfg.img_size,
                               augment=False, cfg=cfg)
    num_classes = full_ds.num_classes
    class_names = [full_ds.gid_to_cat_name[i] for i in range(num_classes)]
    tqdm.write(f"Classes ({num_classes}): {full_ds.gid_to_cat_name}")

    train_ds, val_ds, test_ds = make_train_val_test_splits(
        full_ds, cfg.train_ratio, cfg.val_ratio, cfg.split_seed)
    tqdm.write(f"Split → train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

    model = AgroNet(
        num_classes=num_classes,
        reg_max=cfg.reg_max,
        pretrained_backbone=cfg.pretrained_backbone,
    )

    trainer = AgroNetTrainer(model, train_ds, val_ds, cfg, num_classes, class_names)
    trainer.fit()

    inferencer = AgroNetInferencer(
        model=model,
        checkpoint_path=os.path.join(cfg.log_dir, "checkpoint_best.pth"),
        class_names=class_names,
        cfg=cfg,
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True, collate_fn=okra_collate_fn)

    inferencer.evaluate_dataset(test_loader, prefix="test")
    inferencer.visualize_dataset(test_loader, max_images=50, prefix="test")
    inferencer.gradcam_dataset(test_loader, max_images=20, prefix="test")

In [ ]:
#!rm -r /kaggle/working/